# Generated collection worker 2/3

Generated from `08_collect_verifier_v2_pro.ipynb`; edit the canonical notebook, not this copy.


# 08 — Trajectory-seeded LIBERO-PRO verifier collection

Collect 240 development and 160 blinded confirmatory groups with 12 clean
action candidates each. Apply `supabase/migrations/003_verifier_v2.sql` first.
Use the generated three-worker notebooks for the planned 48 GPU-hour run.

## 1. Setup

In [ ]:
EXTRAS = 'sim,analysis'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Load the simulator, policy, and canonical PRO identities

In [ ]:
import hashlib, json
from collections import defaultdict
from tqdm.auto import tqdm
from pnp import libero_env, libero_pro, models
from pnp.experiments import _prepare_libero_pro_episodes
from pnp.store import SupabaseStore
from pnp.verifier import *

libero_pro.patch_torch_load()
policy, preprocess, postprocess = models.load_pi05()
device = models.default_device(); store = SupabaseStore()
episodes = _prepare_libero_pro_episodes()
for ep in episodes: ep["benchmark"] = "libero_pro"
episode_lookup = {(ep["suite"], ep["task_idx"], ep.get("ep_idx", ep.get("episode_idx"))): ep
                  for ep in episodes}
print({"PRO identities": len(episode_lookup), "device": str(device)})

## 3. Freeze and upload the outcome-blind seeded manifest

In [ ]:
DEVELOPMENT_EXPERIMENT = "verifier-v2-pro-development"
TEST_EXPERIMENT = "verifier-v2-pro-confirmatory"
CANDIDATE_COUNT = 12
PREFIX_LENGTH = 10
SHARD_COUNT = 3   # Generated worker count.
SHARD_INDEX = 2   # Generated worker index.
assert 0 <= SHARD_INDEX < SHARD_COUNT

rollouts = store.fetch_all(
    "rollouts", "rollout_id,benchmark,suite,task_idx,episode_idx,success",
    configure=lambda q: q.eq("experiment", "libero-pro-canonical-core-k3-v1").eq(
        "method", "pnp_uncertainty_only").eq("status", "completed"),
    order_by=("rollout_id",))
rollout_ids = sorted(row["rollout_id"] for row in rollouts)
euler=[]
for start in range(0, len(rollout_ids), 100):
    ids=rollout_ids[start:start+100]
    euler += store.fetch_all(
        "pnp_euler_steps", "rollout_id,chunk_idx,euler_step,u_mean",
        configure=lambda q, ids=ids: q.in_("rollout_id", ids),
        order_by=("rollout_id", "chunk_idx", "euler_step"))
by_rollout={row["rollout_id"]: row for row in rollouts}
uncertainty=defaultdict(list)
for row in euler:
    uncertainty[(row["rollout_id"], int(row["chunk_idx"]))].append(float(row["u_mean"]))
source=[]
for (rollout_id, chunk_idx), values in uncertainty.items():
    if rollout_id in by_rollout:
        source.append({**by_rollout[rollout_id], "chunk_idx": chunk_idx,
                       "u_mean": sum(values)/len(values), "uncertainty_stratum": "high"})
manifests = build_seeded_pro_manifest(
    source, development_target=240, test_target=160, seed=20260728)
hashes={name: collection_manifest_hash(rows) for name, rows in manifests.items()}
document={"version": 3, "candidate_count": CANDIDATE_COUNT,
          "prefix_length": PREFIX_LENGTH, "hashes": hashes, "manifests": manifests}
manifest_path=f"verifier_manifests/verifier-v2-pro-{hashes['development']}-{hashes['confirmatory_test']}.json"
store._upload(manifest_path, json.dumps(document, sort_keys=True,
                                       separators=(",", ":")).encode())
full_manifest=[]
for split, experiment in (("development", DEVELOPMENT_EXPERIMENT),
                          ("confirmatory_test", TEST_EXPERIMENT)):
    full_manifest += [{**row, "experiment": experiment} for row in manifests[split]]
manifest=full_manifest[SHARD_INDEX::SHARD_COUNT]
assert len(full_manifest) == 400
assert len({(r["suite"],r["task_idx"],r["episode_idx"],r["trajectory_seed"])
            for r in full_manifest}) == 400
print({"development": 240, "confirmatory_test": 160, "worker": len(manifest),
       "hashes": hashes, "manifest_path": manifest_path})

## 4. Resume-safe sharded collection

In [ ]:
experiments=(DEVELOPMENT_EXPERIMENT, TEST_EXPERIMENT)
existing_rows=store.fetch_all(
    "verifier_candidate_groups", "candidate_group_id,experiment",
    configure=lambda q: q.in_("experiment", experiments), order_by=("candidate_group_id",))
existing_ids={row["candidate_group_id"] for row in existing_rows}
candidate_rows=[]
for start in range(0, len(existing_ids), 100):
    ids=sorted(existing_ids)[start:start+100]
    candidate_rows += store.fetch_all(
        "verifier_candidates", "candidate_id,candidate_group_id",
        configure=lambda q, ids=ids: q.in_("candidate_group_id", ids), order_by=("candidate_id",))
counts=defaultdict(int)
for row in candidate_rows: counts[row["candidate_group_id"]] += 1
complete={gid for gid in existing_ids if counts[gid] == CANDIDATE_COUNT}
print({"complete_existing_groups": len(complete),
       "partial_groups_to_repair": len(existing_ids-complete)})

store.start_run("verifier_pair_collection", "libero_pro", "verifier-v2-pro",
                config={"candidate_count": CANDIDATE_COUNT, "groups": 400,
                        "manifest_hashes": hashes, "shard_count": SHARD_COUNT,
                        "shard_index": SHARD_INDEX})
new_outcomes=skipped=0
for item in tqdm(manifest, desc="V2 PRO groups"):
    expected=candidate_group_id(
        "libero_pro", item["suite"], item["task_idx"], item["episode_idx"],
        item["chunk_idx"], namespace=item["experiment"],
        trajectory_seed=item["trajectory_seed"])
    if expected in complete: continue
    ep=episode_lookup[(item["suite"], item["task_idx"], item["episode_idx"])]
    env=libero_env.make_env(ep["bddl_path"])
    try:
        try:
            result=collect_replay_candidate_group(
                env, ep, policy, preprocess, postprocess, device,
                chunk_idx=item["chunk_idx"], uncertainty_stratum="high",
                prefix_length=PREFIX_LENGTH, candidate_count=CANDIDATE_COUNT,
                experiment=item["experiment"], trajectory_seed=item["trajectory_seed"],
                collection_split=item["collection_split"],
                manifest_hash=hashes[item["collection_split"]], model_revision="pi05")
        except Exception as error:
            print("group skipped:", type(error).__name__, error); result=None
        if result is None: skipped += 1; continue
        group, candidates=result
        group["metadata_json"].update({
            "collection_manifest_path": manifest_path,
            "source_rollout_id": item["rollout_id"],
            "source_success": bool(item["success"]),
            "source_u_mean": float(item["u_mean"])})
        store.register_candidate_group(group, candidates)
        complete.add(group["candidate_group_id"]); new_outcomes += len(candidates)
    finally: env.close()
store.finish_run(n_rollouts=new_outcomes)
print({"new_outcomes": new_outcomes, "skipped": skipped,
       "complete_groups_seen": len(complete)})

## 5. Integrity report (confirmatory labels remain sealed)

In [ ]:
for split, experiment, target in (
    ("development", DEVELOPMENT_EXPERIMENT, 240),
    ("confirmatory_test", TEST_EXPERIMENT, 160)):
    groups=store.fetch_all("verifier_candidate_groups", "candidate_group_id",
                 configure=lambda q, e=experiment: q.eq("experiment", e), order_by=("candidate_group_id",))
    ids={row["candidate_group_id"] for row in groups}; candidates=[]
    for start in range(0, len(ids), 100):
        batch=sorted(ids)[start:start+100]
        candidates += store.fetch_all("verifier_candidates", "candidate_group_id,success",
                            configure=lambda q, ids=batch: q.in_("candidate_group_id", ids))
    counts=defaultdict(int)
    for row in candidates: counts[row["candidate_group_id"]] += 1
    report={"target": target, "groups": len(ids),
            "complete_groups": sum(counts[gid] == CANDIDATE_COUNT for gid in ids),
            "partial_groups": sum(0 < counts[gid] < CANDIDATE_COUNT for gid in ids)}
    if split == "development":
        outcomes=defaultdict(set)
        for row in candidates: outcomes[row["candidate_group_id"]].add(bool(row["success"]))
        report["discordant_groups"] = sum(len(values)==2 for values in outcomes.values())
    print(split, report)